# HyperAPI Python SDK — a complete walkthrough

One invoice, run through every document operation the platform offers, with the
exact response shape printed at each step.

| # | Operation | Question it answers | Uses an LLM |
|---|-----------|---------------------|-------------|
| 1 | **parse** | What text is on this page? | no — OCR only |
| 2 | **classify** | What kind of document is this? | yes |
| 3 | **split** | Where does one document end and the next begin? | yes |
| 4 | **redact** | Can I share this without exposing personal data? | yes |
| 5 | **extract** | Give me the fields as structured data | yes |
| 6 | **batch** | The same work across many documents, deferred | per endpoint |

### How a call actually works

Every operation follows the same path. The SDK presents it as a single blocking
method:

```
client.extract("invoice.png")
  │
  ├─ POST /v1/documents/upload   →  presigned S3 URL + document_key
  ├─ PUT  <S3 URL>               →  your bytes go straight to S3
  ├─ POST /v1/extract            →  202 { job_id, poll_url }
  ├─ GET  /v1/jobs/{job_id}      →  polled until the job reaches a final state
  └─ returns the completed response
```

Because each individual HTTP request completes in well under a second, the call
is never subject to a CDN or load-balancer idle timeout — even when the job
itself takes minutes to finish.

### Before you begin

You need an API key and the sample document. The notebook makes roughly a dozen
API calls; on the free tier, which permits one submission per minute, expect it
to pace itself accordingly (this is handled for you in §4).

---
## 1 · Install the SDK

The SDK is distributed from GitHub rather than PyPI, so `pip install hyperapi`
will not resolve. Install from the repository directly:

```
pip install "hyperapi @ git+https://github.com/hyprbots/hyperapi-sdk.git"
```

> **The public build reports its version as `0.0.0`.** Release labelling is
> stripped from the published SDK, so the version string is not a useful
> indicator of what you have. The cell below therefore confirms the build by
> checking that the expected methods are present, rather than by comparing
> version numbers.

In [ ]:
import importlib
import subprocess
import sys
import sysconfig
from pathlib import Path

GIT_SPEC = "hyperapi @ git+https://github.com/hyprbots/hyperapi-sdk.git"

# The operations this notebook walks through. Checking for these is more
# meaningful than a version comparison, given the unlabelled public build.
EXPECTED_METHODS = (
    "parse", "classify", "split", "redact", "extract", "extract_advanced",
    "upload_document", "create_batch", "wait_for_batch", "download_pages",
)


def install_sdk(source=GIT_SPEC):
    """Install or upgrade the SDK. Pass a local path to install a checkout."""
    print(f"Installing: {source}\n")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", str(source)],
        check=True,
    )
    importlib.invalidate_caches()      # let the running kernel see the new package


try:
    import hyperapi
except ImportError:
    install_sdk()
    import hyperapi

print(f"hyperapi {hyperapi.__version__}")
print(f"loaded from {Path(hyperapi.__file__).parent}\n")

# If an editable install or a directory on sys.path shadows the published
# package, `import hyperapi` succeeds without this cell ever installing
# anything — and you would demonstrate a local working copy while believing
# it to be the released SDK. Say so plainly rather than let it pass silently.
site_packages = sysconfig.get_paths().get("purelib", "")
if site_packages and not str(Path(hyperapi.__file__).parent).startswith(site_packages):
    print("Note: this is a local checkout, not the published package.")
    print("      Run install_sdk() and restart the kernel to use the released SDK.\n")

missing = [m for m in EXPECTED_METHODS if not hasattr(hyperapi.HyperAPIClient, m)]
if missing:
    print(f"Missing expected methods: {', '.join(missing)}")
    print("Run install_sdk(), then restart the kernel.")
else:
    print(f"All {len(EXPECTED_METHODS)} expected operations are available.")

# Pillow is used only to display pages inside the notebook.
try:
    import PIL  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pillow"], check=True)

---
## 2 · Configure

Set your API key below, or export `HYPERAPI_KEY` before starting Jupyter.

| Key prefix | Purpose |
|---|---|
| `hk_live_…` | production traffic — billed, counts against your quota |
| `hk_test_…` | test traffic — excluded from billing dashboards |
| `hk_service_…` | service account, for machine-to-machine use |

The base URL defaults to `https://apis.hyperbots.com`. Note the **`apis`**
subdomain: `api.hyperbots.com` is a different host and does not serve `/v1/*`.

In [ ]:
import base64
import json
import os
import textwrap
import time
from pathlib import Path

# These five are used throughout the notebook, not just in this cell.

# ── Configure ─────────────────────────────────────────────────────────────
API_KEY  = os.environ.get("HYPERAPI_KEY", "")          # your hk_live_… key
BASE_URL = os.environ.get("HYPERAPI_URL", "https://apis.hyperbots.com")
DOCUMENT = Path("images/spectrum_remit.png")   # ships with this notebook
# ──────────────────────────────────────────────────────────────────────────

# Check both up front, so a missing key surfaces here rather than as a
# confusing 401 several cells later.
assert API_KEY, "Set API_KEY above, or export HYPERAPI_KEY, before continuing."
assert DOCUMENT.exists(), f"Document not found: {DOCUMENT}"

print(f"key       hk_***{API_KEY[-4:]}")
print(f"base_url  {BASE_URL}")
print(f"document  {DOCUMENT.name}  ({DOCUMENT.stat().st_size / 1024:.0f} KB)")

---
## 3 · The sample document

A single-page invoice from a medical imaging provider. Two characteristics are
worth noting now, because they explain results further down:

- **The filename says `remit`, but the document is an invoice.** It carries an
  invoice number, a line-item grid and a balance due. `classify` reads the
  document itself, so it will report `invoice`.
- **It is one page.** `split` is designed for multi-document PDFs, so it will
  report that no split was required — the correct answer for this input, and a
  different response shape from the segmented case. See §7.

In [ ]:
from PIL import Image
from IPython.display import display

document_image = Image.open(DOCUMENT)
print(f"{document_image.size[0]} × {document_image.size[1]} px · mode {document_image.mode}")
display(document_image.resize((document_image.width // 2, document_image.height // 2)))

---
## 4 · Helper functions

Three small utilities, defined once so the sections that follow stay focused on
the API rather than on presentation.

**Rate limiting.** The API gateway enforces a per-organization sliding window on
every `/v1/*` submission:

| Plan | Submissions per minute |
|---|---|
| Free | 1 |
| Pro | 60 |
| Enterprise | 1000 |

Running a dozen operations back to back will exceed that on a free key. The SDK
already backs off automatically while *polling* a job, but a rejected
*submission* raises `RateLimitError` to the caller. The `run()` helper catches
it and waits for the interval the server specifies.

**Output formatting.** `describe()` prints the shape of a response — keys and
types — rather than dumping the full payload. `table()` renders aligned rows
without requiring pandas.

In [ ]:
from hyperapi import HyperAPIClient, RateLimitError, HyperAPIError, JobTimeoutError

TIMINGS = {}


def run(label, fn, *args, **kwargs):
    """Call an SDK method, handling rate-limit rejections and recording elapsed time.

    The server states exactly when the rate-limit window reopens, so `retry_after`
    is honoured directly rather than approximated with a backoff curve.
    """
    started = time.monotonic()
    for attempt in range(1, 6):
        try:
            result = fn(*args, **kwargs)
        except RateLimitError as e:
            wait = e.retry_after + 1
            detail = " (rate limiter degraded; transient)" if e.degraded else ""
            print(f"  Rate limited on {label}, plan={e.tier}{detail}. "
                  f"Retrying in {wait}s (attempt {attempt} of 5).")
            time.sleep(wait)
            continue
        elapsed = time.monotonic() - started
        TIMINGS[label] = elapsed
        print(f"  {label} completed in {elapsed:.1f}s")
        return result
    raise RuntimeError(f"{label}: still rate limited after 5 attempts.")


def describe(obj, depth=2, width=100):
    """Print a response's structure — keys, types and sizes — not its full contents."""
    def walk(value, remaining, indent=""):
        if isinstance(value, dict):
            for key, item in value.items():
                kind = type(item).__name__
                if isinstance(item, (dict, list)) and remaining > 0:
                    size = f"[{len(item)}]" if isinstance(item, list) else f"{{{len(item)}}}"
                    print(f"{indent}{key}: {kind}{size}")
                    nested = item[0] if isinstance(item, list) and item else item
                    walk(nested, remaining - 1, indent + "  ")
                else:
                    flat = str(item).replace("\n", " / ")
                    print(f"{indent}{key}: {kind} = {textwrap.shorten(flat, width)}")
        elif isinstance(value, list) and value and remaining > 0:
            print(f"{indent}first element:")
            walk(value[0], remaining - 1, indent + "  ")
    walk(obj, depth)


def table(rows, headers):
    """Render aligned rows. Short rows are padded rather than raising, so a
    response missing an optional field cannot interrupt the walkthrough."""
    n = len(headers)
    rows = [[("" if c is None else str(c)) for c in r][:n] + [""] * (n - len(r))
            for r in rows]
    widths = [max([len(h)] + [len(r[i]) for r in rows]) for i, h in enumerate(headers)]
    print("  ".join(h.ljust(w) for h, w in zip(headers, widths)))
    print("  ".join("-" * w for w in widths))
    for r in rows:
        print("  ".join(c.ljust(w) for c, w in zip(r, widths)))


def show_image(source, width=560):
    """Display an image file at a readable width, preserving aspect ratio."""
    image = Image.open(source) if not isinstance(source, Image.Image) else source
    height = round(width * image.height / image.width)
    display(image.resize((width, height)))

### Creating the client

`HyperAPIClient` is **not** thread-safe — nor is the `httpx.Client` it wraps.
Use one client per thread, or a `threading.local()` instance.

Two constructor settings are worth knowing:

- **`poll_interval`** — seconds between job-status checks. Defaults to 3.
- **`poll_timeout`** — the total time the SDK will wait for a single job.
  Defaults to 3600 seconds. This is intentionally longer than the web
  playground's 30-minute limit, on the basis that a script can afford to wait
  where an interactive session cannot. A long-running document may therefore
  time out in the playground yet complete successfully through the SDK.

The client is also a context manager (`with HyperAPIClient(...) as client:`),
which closes the connection pool automatically.

In [ ]:
client = HyperAPIClient(api_key=API_KEY, base_url=BASE_URL, poll_interval=3.0)
print(client)      # the repr masks the key

---
## 5 · `parse` — optical character recognition

`parse` is the only operation that returns OCR output directly. The others take
that same OCR result and pass it to a language model for further processing.

```
parse                            OCR  ─────────────────────▶  text

classify · split · extract · redact
                                 OCR  ──▶  language model  ──▶  structured result
```

### Choosing an OCR engine

| Call | Engine used |
|---|---|
| `parse(doc)` | your organization's configured default (platform default: fast) |
| `parse(doc, mode="fast")` | standard OCR — plain text, optional word boxes |
| `parse(doc, mode="advanced")` | layout-aware — adds `structured` per page: `html`, `markdown`, `regions` |

> **`mode` defaults to `None`, not `"fast"`.** When omitted, the SDK leaves the
> parameter out of the request so your organization's own default applies.
> Passing `"fast"` explicitly *overrides* that configuration. Omit `mode` unless
> you intend to override it. The same applies to `parse_mode` on `extract()`
> and `extract_advanced()`.

**Where the result lives:** nested one level below `result` — OCR text at
`result["result"]["ocr"]`, pages at `result["result"]["pages"]`.

> **Large advanced documents are the exception.** A document above the 60-page
> cap for advanced parse — where the deployment permits it — returns no inline
> `pages` or `ocr`. The payload sits behind a presigned `result["result_url"]`,
> and any page ranges that failed are listed in `result["metadata"]["gaps"]`.
> Test for `result_url` rather than assuming the inline shape, and raise
> `poll_timeout`, since such a job can exceed the 3600-second default. Our
> single-page sample always returns the inline form.

In [ ]:
# No `mode` argument: this respects whatever OCR profile the organization
# is configured to use.
parsed = run("parse (organization default)", client.parse, DOCUMENT)

print("\nResponse structure")
print("------------------")
describe(parsed, depth=2)

# Branch on `result_url` rather than assuming the inline shape — a large
# advanced document returns the payload by reference instead. See the note above.
if parsed.get("result_url"):
    print("\nLarge-document response: the payload is at parsed['result_url'].")
    print(f"Failed page ranges: {(parsed.get('metadata') or {}).get('gaps')}")
else:
    ocr_text = parsed["result"].get("ocr") or ""
    print(f"\nOCR text ({len(ocr_text)} characters)")
    print("------------------")
    print(ocr_text[:900])

### Word boxes and page images

Two optional arguments enrich each page:

- **`include_boxes=True`** adds a `boxes` list, each entry being
  `{"text", "bbox": [left, top, right, bottom], "confidence"}`.
- **`include_image=True`** adds `image_url`, a presigned link to the
  deskew-corrected page, along with `dimensions` — the pixel space the box
  coordinates are expressed in.

Two constraints to be aware of:

- Word boxes are produced by the **standard** engine only. `mode="advanced"`
  returns an empty `boxes` list and provides `structured` instead. This is the
  one situation where passing `mode="fast"` explicitly is appropriate.
- `image_url` is a presigned S3 link valid for roughly 15 minutes, while the job
  itself is retained for 24 hours. If a link expires, call `get_job(job_id)` to
  obtain fresh URLs rather than re-running the parse.

In [ ]:
detailed = run(
    "parse (boxes and images)", client.parse,
    DOCUMENT,
    mode="fast",            # explicit override: only this engine produces boxes
    include_boxes=True,
    include_image=True,
)

pages = detailed["result"].get("pages") or []
first_page = pages[0] if pages else {}
print(f"\npages returned : {len(pages)}")
print(f"page keys      : {sorted(first_page)}")
print(f"dimensions     : {first_page.get('dimensions')}")
print(f"boxes detected : {len(first_page.get('boxes') or [])}\n")

table(
    [[b["text"][:38], [round(v) for v in b["bbox"]], round(b.get("confidence", 0), 3)]
     for b in (first_page.get("boxes") or [])[:10]],
    ["text", "bbox [left, top, right, bottom]", "confidence"],
)

`download_pages()` writes every page image in a response to disk. It looks for a
`pages` list containing `image_url`, which `parse(include_image=True)`,
`edit_detect()` and `edit_fill()` all return. Note that `redact()` does **not**
return that structure — see §8.

In [ ]:
output_dir = Path("downloaded_pages")
saved_pages = client.download_pages(detailed, output_dir, prefix="parsed")

print(f"Wrote {len(saved_pages)} page image(s) to {output_dir}")
show_image(saved_pages[0])

### Advanced parse — layout-aware OCR

`mode="advanced"` uses the layout-aware engine, which reconstructs reading
order, tables and headings instead of producing a flat stream of text. It is
slower, and on some plans it is restricted to paid tiers.

For a table-heavy document such as this invoice, the `markdown` output is
generally the most useful representation: it preserves the line-item grid that
plain OCR flattens into ambiguous whitespace.

In [ ]:
try:
    advanced = run("parse (advanced)", client.parse, DOCUMENT, mode="advanced")
    advanced_pages = advanced["result"].get("pages") or []
    structured = ((advanced_pages[0] if advanced_pages else {}) or {}).get("structured") or {}
    print(f"\nstructured keys: {sorted(structured)}\n")
    print("Markdown")
    print("--------")
    print((structured.get("markdown") or "(none returned)")[:1200])
except HyperAPIError as e:
    # Advanced parse is restricted on some plans. A 403 here indicates a
    # plan boundary, not a problem with the document.
    print(f"Advanced parse unavailable: HTTP {e.status_code} — {e.message}")
    print(f"Request ID for support: {e.request_id}")

---
## 6 · `classify` — identifying the document type

This is where the filename becomes instructive. The file is named
`spectrum_remit.png`, but its contents are an invoice. The classifier evaluates
the document, not its name, and should return `invoice`.

**Where the result lives:** `result["result"]["document_type"]`, accompanied by
`confidence`, `domain` and `needs_review`. The raw OCR text sits at the top
level of the response, in `result["ocr_text"]`.

> `confidence` is a **categorical string** — `"very_high"`, `"high"`, and so on
> — not a number. Comparing it numerically raises a `TypeError`.

In [ ]:
classified = run("classify", client.classify, DOCUMENT)

classification = classified["result"]
table(
    [[field, repr(classification.get(field))]
     for field in ("document_type", "confidence", "domain", "needs_review")],
    ["field", "value"],
)

print(f"\nFilename suggests : {DOCUMENT.stem}")
print(f"Document is       : {classification.get('document_type')}")

### Restricting the label set

By default the classifier selects from a broad built-in taxonomy. In production
you generally know which document types your pipeline can handle, and declaring
them improves accuracy while ensuring no unexpected label reaches your routing
logic.

`options` accepts the following settings:

| Key | Effect |
|---|---|
| `mode` | `"fast"`, `"balanced"` or `"thorough"` — pipeline depth |
| `active_classes` | restrict classification to an explicit list of labels |
| `active_classes_type` | `"default"` (full built-in set) or `"minimal"` |
| `custom_llm_classes` | add your own labels |
| `system_instruction` | additional prompt guidance |
| `confusion_neighbors` | `{label: [neighbour, …]}` hints for easily confused types |
| `start_token_budget`, `max_start_pages` | how much of the document to read |

Note that `options["mode"]` sets pipeline depth, whereas the `mode=` argument
selects the task. They are unrelated settings that share a name.

In [ ]:
constrained = run(
    "classify (restricted labels)", client.classify, DOCUMENT,
    options={
        "mode": "balanced",
        "active_classes": ["invoice", "remittance_advice", "purchase_order", "receipt"],
        "confusion_neighbors": {"invoice": ["remittance_advice"]},
    },
)

restricted = constrained["result"]
print(f"{restricted.get('document_type')}  (confidence: {restricted.get('confidence')})")

---
## 7 · `split` — finding document boundaries

`split` addresses the case where a single 200-page PDF actually contains 40
separate invoices. It identifies the boundaries between them and labels each
segment.

**This operation returns two different shapes, and you must handle both.**

| Case | Response |
|---|---|
| Boundaries found | `result["result"]["segmentation"]["splits"]` — a list of segments |
| Document too short to split | `result["result"]` has **`not_required: true`** and a `reason`, and **no `segmentation` key at all** |

Our sample is a single page, so it takes the second path. Indexing straight into
`["segmentation"]` raises `KeyError` — which is exactly why the code below tests
for the key rather than assuming it.

Each segment, when present, contains `start_page`, `end_page`, `confidence`, a
label (`split_label`, sometimes `label`) and often `identifying_features`.

> Note that `confidence` here is a **number**, unlike `classify`, where it is a
> categorical string. The two operations do not agree on this.

In [ ]:
split_result = run("split", client.split, DOCUMENT)

payload = split_result["result"]
print(f"\nresult keys: {sorted(payload)}\n")

# Either shape may arrive; read both defensively. This mirrors the logic the
# dashboard itself uses to render split results.
segmentation = payload.get("segmentation") or {}
splits = segmentation.get("splits") or []
not_required = payload.get("not_required") is True or segmentation.get("not_required") is True
reason = payload.get("reason") or segmentation.get("reason")

if not_required or not splits:
    print("No split required — the document was kept whole.")
    if reason:
        print(f"Reason: {reason}")
else:
    table(
        [[s.get("split_label") or s.get("label"),
          s.get("start_page"), s.get("end_page"), s.get("confidence"),
          textwrap.shorten(str(s.get("identifying_features") or "-"), 40)]
         for s in splits],
        ["label", "start page", "end page", "confidence", "identifying features"],
    )
    print(f"\n{len(splits)} segment(s) found.")

---
## 8 · `redact` — removing personal data

Two modes share a single detection pass:

| `mode` | Result |
|---|---|
| `"redact"` (default) | personal data is covered with black boxes |
| `"deidentify"` | personal data is replaced with realistic synthetic values, leaving the document readable and structurally intact |

The built-in categories are `PERSON_NAME`, `COMPANY_NAME`, `EMAIL`, `ADDRESS`,
`WEBSITE`, `PHONE` and `CREDENTIALS` (passwords, API keys and tokens). Each has
a synthetic replacement defined, so every category works in both modes. Setting
`include_logos=True` additionally detects and masks logos.

**Where the result lives — and how it differs.** `redact` returns
`result["result"]["images"]`, a list of base64-encoded page images, together
with `updated_text_block` and a `summary` of how many items were masked per
category. It does **not** return a `pages` list, so `download_pages()` does not
apply here. The helper below accepts either a base64 payload or a URL, since
both forms occur depending on how the result is retrieved.

In [ ]:
from IPython.display import Image as DisplayImage


def show_redacted(images, limit=3):
    """Display redaction output, whether returned as base64 data or as a URL."""
    for index, entry in enumerate(images[:limit], start=1):
        if not isinstance(entry, str):
            print(f"page {index}: unexpected type {type(entry).__name__}")
            continue
        if entry.startswith(("http://", "https://")):
            print(f"page {index} (presigned URL)")
            display(DisplayImage(url=entry))
        else:
            encoded = entry.split(",", 1)[-1]      # tolerate a data: URI prefix
            print(f"page {index} ({len(encoded)} base64 characters)")
            display(DisplayImage(data=base64.b64decode(encoded)))


redacted = run("redact", client.redact, DOCUMENT, mode="redact")

redaction = redacted["result"]
print(f"\nresult keys: {sorted(redaction)}\n")

masked_counts = redaction.get("summary") or {}
if masked_counts:
    table([[k, v] for k, v in masked_counts.items()], ["category", "items masked"])
else:
    print("No summary of masked categories was returned.")

print()
show_redacted(redaction.get("images") or [])

### Deidentification and custom categories

`mode="deidentify"` keeps the document usable. An invoice with a black rectangle
covering the total is of little use for testing a downstream parser; one showing
a different but plausible total remains a valid test fixture.

`pii_config` adjusts which categories are detected:

- `{"mode": "extend", "types": [...]}` — retain the built-in categories and add
  your own
- `{"mode": "replace", "types": [...]}` — detect only the categories you list

In [ ]:
deidentified = run(
    "deidentify", client.redact, DOCUMENT,
    mode="deidentify",
    pii_config={"mode": "extend", "types": [{"name": "MEDICAL_RECORD_NUMBER"}]},
)

deidentification = deidentified["result"]
print(f"\nsummary: {deidentification.get('summary')}\n")

substituted_text = deidentification.get("updated_text_block")
if substituted_text:
    print("Text after substitution")
    print("-----------------------")
    print(str(substituted_text)[:700], "\n")

show_redacted(deidentification.get("images") or [])

---
## 9 · `extract` — structured field extraction

Three distinct extractors are available through this method. Selecting the wrong
one is the most common source of confusion with this API.

| Call | Detects document type | Appropriate when |
|---|---|---|
| `extract(category="financial")` *(default)* | no — assumes the invoice family | processing invoices, receipts and bills |
| `extract(category="non_financial", schema=…)` | no — assumes a single document type | you know the shape you want returned |
| `extract_advanced()` | **yes** | documents of mixed or unknown type |

### Financial extraction — the default

This routes to the invoice adapter, which divides the result three ways:

| Path | Contents |
|---|---|
| `result["result"]["entities"]` | header and party fields — vendor, invoice number, dates |
| `result["result"]["line_items"]` | one entry per line |
| `result["result"]["summary"]` | document-level totals — tax, currency, grand total |

> The grand total is found in `summary["total_amount"]`, not in `entities`.

In [ ]:
extracted = run("extract (financial)", client.extract, DOCUMENT, category="financial")

extraction = extracted["result"]
print(f"\nresult keys: {sorted(extraction)}\n")

print("Entities")
print("--------")
entities = extraction.get("entities") or {}
table([[k, textwrap.shorten(str(v), 60)] for k, v in entities.items()],
      ["field", "value"])

print("\nLine items")
print("----------")
line_items = extraction.get("line_items") or []
if line_items:
    columns = list(line_items[0].keys())
    table([[item.get(c) for c in columns] for item in line_items], columns)
else:
    print("(none)")

print("\nSummary — document-level totals")
print("-------------------------------")
table([[k, v] for k, v in (extraction.get("summary") or {}).items()],
      ["field", "value"])

### Verifying the result

The invoice states 35 × 65.00 = 2,275.00, with both the total and the balance
due shown as $2,275.00. Reconciling extracted figures against arithmetic you can
verify independently is an inexpensive quality check, and a natural candidate
for an automated test.

In [ ]:
summary_fields = extraction.get("summary") or {}


def to_amount(value):
    """Convert '$2,275.00' to 2275.0. Returns None if the value cannot be parsed."""
    if value is None:
        return None
    try:
        return float(str(value).replace("$", "").replace(",", "").strip())
    except ValueError:
        return None


# Filter on `is not None` rather than truthiness, so a legitimate 0.00 counts.
amounts = [to_amount(item.get("amount") or item.get("total")) for item in line_items]
line_item_total = sum(a for a in amounts if a is not None)
stated_total = to_amount(summary_fields.get("total_amount"))
expected_total = 2275.00

print(f"Line items sum to : {line_item_total:,.2f}")
print(f"Reported total    : {'not found' if stated_total is None else format(stated_total, ',.2f')}")
print(f"Document states   : {expected_total:,.2f}")
print()
if stated_total is not None and abs(stated_total - expected_total) < 0.01:
    print("Match: the extracted total agrees with the document.")
else:
    print("Mismatch: review the raw response before relying on these figures.")

### Non-financial extraction — supplying your own shape

`category="non_financial"` performs no document-type detection at all. You
provide a blank template, and the response returns that same structure populated
from the document.

The template is a JSON object using `None`, `False` or `"unselected"` as
placeholders for the values you want back. The `schema` argument accepts a
`dict`, a path to a `.json` file, or a JSON string.

Nothing is stored on the server. Supplying a different `schema` on the next call
simply returns a different shape — there is no merging, versioning or persistent
state involved.

**Where the result lives:** `result["result"]["data"]`, matching your template
rather than the `entities` / `line_items` / `summary` division above.

In [ ]:
TEMPLATE = {
    "vendor": {"name": None, "street": None, "city_state_zip": None},
    "bill_to": {"name": None, "street": None, "city_state_zip": None},
    "invoice_number": None,
    "invoice_date": None,
    "services": [{"description": None, "quantity": None, "rate": None, "amount": None}],
    "total": None,
    "balance_due": None,
    "payable_to": None,
    "is_past_due": False,          # a boolean placeholder
}

templated = run(
    "extract (non-financial)", client.extract, DOCUMENT,
    category="non_financial", schema=TEMPLATE,
)

print("\nCompleted template")
print("------------------")
print(json.dumps(templated["result"].get("data"), indent=2)[:1800])

### Advanced extraction — automatic type detection

`extract_advanced()` takes no `category` argument. It identifies the document
first and then routes accordingly: invoice-family documents go to the invoice
adapter, and everything else to schema-less grounded extraction. This is the
appropriate choice for an inbound queue of mixed document types.

In [ ]:
try:
    auto = run("extract_advanced", client.extract_advanced, DOCUMENT)
    print(f"\nresult keys: {sorted(auto['result'])}\n")
    describe(auto["result"], depth=2)
except HyperAPIError as e:
    print(f"Unavailable: HTTP {e.status_code} — {e.message} (request ID: {e.request_id})")

---
## 10 · Batch processing

Every operation so far has been interactive: submit, wait, receive a result.
Batch processing inverts that model. You supply a list of previously uploaded
documents and receive a `batch_id` immediately; the work then runs on spare
capacity, yielding to live traffic.

```
Interactive   submit ──▶ poll ──▶ result              (seconds to minutes)

Batch         submit ──▶ batch_id                     (returns immediately)
                          └─▶ workers process the queue   (24-hour SLA)
                          └─▶ poll, or receive a webhook  (at your convenience)
```

Supported endpoints are `/v1/parse`, `/v1/extract`, `/v1/classify`, `/v1/split`
and `/v1/redact`.

**Uploading separately.** `upload_document()` returns a `document_key` that can
be reused across any number of calls without resending the file, and that is
what a batch consumes. `create_batch_from_files()` combines the two steps, but
performing them separately makes the mechanism clearer — and lets us build a
two-item batch from a single upload.

In [ ]:
# A single upload produces a reusable key. In practice these would be distinct
# documents; repeating one keeps the walkthrough inexpensive while still
# demonstrating per-item accounting.
document_key = run("upload_document", client.upload_document, DOCUMENT)
print(f"\ndocument_key: {document_key}\n")

batch = run(
    "create_batch", client.create_batch,
    endpoint="/v1/classify",
    document_keys=[document_key, document_key],
    estimated_pages_per_doc=1,
    metadata={"demo": "sdk-walkthrough", "source": DOCUMENT.name},
    # Resubmitting with the same idempotency key returns the existing batch
    # rather than creating a duplicate, making retries safe.
    idempotency_key=f"sdk-walkthrough-{DOCUMENT.stem}-v1",
)
print(json.dumps(batch, indent=2))

### Waiting for completion

`wait_for_batch()` polls until the batch reaches a final state: `completed`,
`completed_with_errors`, `failed` or `canceled`. If the timeout elapses it
raises `JobTimeoutError`, but the batch continues running on the server — you
can resume with `get_batch()` rather than resubmitting.

A short timeout is used here to keep the walkthrough moving; a production
workload would allow hours.

In [ ]:
try:
    completed = run("wait_for_batch", client.wait_for_batch,
                    batch["batch_id"], timeout=600, poll_interval=5)
except JobTimeoutError as e:
    print(f"Still running after {e.elapsed_s:.0f}s. Batch {e.job_id} is unaffected.")
    completed = client.get_batch(batch["batch_id"])

print(f"\nstatus : {completed.get('status')}")
print(f"counts : {completed.get('counts')}\n")

table(
    [[item.get("doc_index"), item.get("status"),
      (item.get("result_key") or "-")[-44:],
      "yes" if item.get("result_url") else "no",
      textwrap.shorten(item.get("error") or "-", 30)]
     for item in completed.get("items", [])],
    ["index", "status", "result key (last 44 chars)", "URL provided", "error"],
)

### Retrieving batch results

Each completed item carries a `result_key` — the storage location of that
document's result — and, where available, a presigned `result_url`.

> `result_url` is provided on a best-effort basis. It is populated only when the
> service has credentials for the results bucket; otherwise only `result_key` is
> returned and you retrieve the object using your own credentials. Test for
> `None` rather than assuming the URL is present.

In [ ]:
import httpx

finished = [i for i in completed.get("items", []) if i.get("status") == "done"]
print(f"{len(finished)} item(s) completed\n")

for item in finished[:2]:
    url = item.get("result_url")
    if not url:
        print(f"Item {item['doc_index']}: no presigned URL provided. "
              f"Retrieve {item.get('result_key')} using your own credentials.")
        continue
    response = httpx.get(url, follow_redirects=True, timeout=60)
    response.raise_for_status()
    print(f"Item {item['doc_index']}")
    print("-------")
    describe(response.json(), depth=2)
    print()

### Managing batches

```python
client.list_batches(limit=20)     # your batches, newest first
client.get_batch(batch_id)        # a single status check
client.cancel_batch(batch_id)     # stop queued items; in-flight items finish
```

`create_batch()` also accepts `webhook_url=` for completion notification, and
`parse_mode="advanced"` for layout-aware batch parsing. The latter requires a
paid plan; the API returns 403 otherwise.

In [ ]:
recent = client.list_batches(limit=5)

table(
    [[str(b.get("batch_id"))[:8], b.get("status"), b.get("endpoint"),
      b.get("total_items"), b.get("done_items"), b.get("failed_items"),
      str(b.get("created_at"))[:19]]
     for b in recent.get("batches", [])],
    ["batch", "status", "endpoint", "items", "done", "failed", "created"],
)

---
## 11 · Reference summary

### Where each result is located

The nesting is not uniform across operations:

| Operation | Path to the result |
|---|---|
| `parse` | `r["result"]["ocr"]` · `r["result"]["pages"]` |
| `parse` (large advanced) | `r["result_url"]` *(presigned)* · `r["metadata"]["gaps"]` |
| `classify` | `r["result"]["document_type"]` · `["confidence"]` *(string)* |
| `split` (boundaries found) | `r["result"]["segmentation"]["splits"]` — one level deeper |
| `split` (too short to split) | `r["result"]["not_required"]` · `["reason"]` — **no `segmentation` key** |
| `redact` | `r["result"]["images"]` *(base64)* · `["summary"]` |
| `extract(financial)` | `r["result"]["entities"]` · `["line_items"]` · `["summary"]["total_amount"]` |
| `extract(non_financial)` | `r["result"]["data"]` — your template, populated |
| batch item | `item["result_url"]` *(may be `None`)* · `item["result_key"]` |

For `classify` and `extract`, the raw OCR text is at the top level of the
response, in `r["ocr_text"]`.

### Defaults worth confirming

| Parameter | Default | Meaning |
|---|---|---|
| `parse(mode=…)` | `None` | defer to the organization's OCR profile — not `"fast"` |
| `extract(parse_mode=…)` | `None` | as above |
| `classify(mode=…)`, `split(mode=…)` | `"default"` | task selector, unrelated to the OCR engine |
| `redact(mode=…)` | `"redact"` | as opposed to `"deidentify"` |
| `extract(category=…)` | `"financial"` | assumes the invoice family; performs no type detection |

### Synchronous and asynchronous use

Each convenience method — `parse`, `classify` and the rest — is a `submit_*`
call followed by `wait_for_job` internally. Separating them allows you to submit
work and collect it later:

```python
job = client.submit_extract("invoice.pdf")     # returns immediately
print(job.job_id, job.status)
...                                            # do other work
result = client.wait_for_job(job)              # block when ready
```

`wait_for_jobs([...])` polls several jobs together, and `client.process(file)`
runs parse and extract concurrently from a single upload, returning
`{"ocr": …, "data": …}` in approximately the time of the slower of the two
rather than their combined duration.

A fully asynchronous `AsyncHyperAPIClient` mirrors every method shown here.

In [ ]:
print("Elapsed time by operation")
print("-------------------------")
table(
    [[label, f"{seconds:6.1f}s"]
     for label, seconds in sorted(TIMINGS.items(), key=lambda kv: -kv[1])],
    ["operation", "elapsed"],
)
print(f"\nTotal: {sum(TIMINGS.values()):.1f}s across {len(TIMINGS)} calls")

client.close()      # releases the connection pool
print("Client closed.")

---
## 12 · Troubleshooting

Every exception carries `status_code` and `request_id`. Quote the `request_id`
when contacting support — server-side logs are indexed by it.

```python
from hyperapi import HyperAPIError

try:
    client.extract("invoice.pdf")
except HyperAPIError as e:
    print(e.status_code, e.request_id, e.message)
```

| Symptom | Cause | Resolution |
|---|---|---|
| `pip install hyperapi` reports no matching distribution | the SDK is distributed from GitHub, not PyPI | install from the repository — see §1 |
| `pip install …git@<branch>` fails at `git checkout` | the repository publishes a single branch, `main` | omit the `@branch` suffix entirely |
| `hyperapi.__version__` reports `0.0.0` | release labelling is stripped from the public build | expected; check for the methods you need instead |
| `parse` result has no `pages` or `ocr` | a large advanced document returned a presigned payload | read `result["result_url"]`; check `result["metadata"]["gaps"]` |
| `AuthenticationError` (401) | incorrect key or host | confirm the key and that the host is `apis.hyperbots.com` |
| `RateLimitError` (429) | free plan permits one submission per minute | wait `e.retry_after` seconds; `e.degraded=True` indicates a transient condition |
| `JobTimeoutError` | the SDK stopped waiting; the job did not stop | resume with `get_job(e.job_id)`, or raise `poll_timeout` |
| 403 on `mode="advanced"` | layout-aware parse requires a paid plan | use `mode="fast"`, or upgrade |
| 403 when downloading a page image | the presigned URL expired (~15 minutes) | call `get_job(job_id)` for fresh URLs |
| `download_pages()` reports no `pages` in the result | `redact` returns `images`, not `pages` | decode the base64 values in `result["images"]` |
| batch `result_url` is `None` | the service has no credentials for the results bucket | retrieve `result_key` using your own credentials |
| `TypeError` when comparing `confidence` | it is a string such as `"very_high"` | compare against the string labels |
| OCR output differs between environments | omitting `mode` defers to the organization profile | pass `mode="fast"` or `"advanced"` to fix the engine |
| `KeyError: 'segmentation'` from `split` | a document too short to split returns `not_required` instead | test for the key; see §7 |
| `TypeError` comparing split `confidence` | numeric in `split`, a string in `classify` | the two operations differ; check the type |

### Exception hierarchy

```
HyperAPIError                      catch this to handle every error type
├── AuthenticationError            401
├── RateLimitError                 429 — .retry_after .tier .limit .degraded
├── JobTimeoutError                    — .job_id .elapsed_s
├── DocumentUploadError            the upload or storage step failed
└── ParseError · ExtractError · ClassifyError · SplitError · RedactError · EditError
```

API keys are removed from exception messages before they are raised, so error
text is safe to log.

### Enabling SDK logging

```python
import logging

logging.basicConfig(level=logging.INFO)
logging.getLogger("hyperapi").setLevel(logging.INFO)
```

The SDK installs a `NullHandler`, so it produces no output until you configure
logging explicitly.

### Not covered here

The `edit` API — form field detection and completion, via `client.edit_detect()`
and `client.edit_fill()` — and the asynchronous `AsyncHyperAPIClient`.